# Agentic_RL_Exec: GSM8K GRPO control arm (Qwen2.5-0.5B, 100 step, tool_agent-no-tools)

**할 일(사람)**: 런타임을 A100으로 설정 → Run all → **1분 내** Drive 인증 팝업에서 "Allow" 클릭.

**중간 개입 필요(1회)**: 학습 셀이 step5 근처에서 `timing_s/step`을 출력하면 로그를 보고
100 step 전체 예상 소요시간을 계산해서 알려줌 — 그대로 계속 둘지 판단 후 별다른 조치
없이 두면 자동으로 100step까지 완주됨(중단하고 싶으면 셀 실행 중지).

이 노트북은 브라우저에서 사람이 Run all로 직접 돌리는 걸 전제로 함(colab-cli 세션 소실
버그 #106 회피 — 상세: vault Colab_Ops-MOC). Drive에 결과 로그를 저장해 세션이 죽어도
산출물이 남게 함.

**목적**: no-tool arm이지만 tool arm(추후 별도 노트북)과 agent_loop 종류를 통일해
confound를 없앤 control — `default_agent_loop=tool_agent`를 tools 없이 사용(전역
기본값 `single_turn_agent`를 쓰면 agent_loop 클래스 자체가 tool arm과 달라져
비교 불가, 상세: `~/workspace/agentic_rl/JOURNAL.md` 2026-09-27 항목).
`num_turns/mean`이 정확히 2.0(user+assistant)이어야 정상 — 그 이상이면 모델이
tool_call 태그를 환각 생성해 "Unknown function" 응답을 받고 있다는 신호.
</cell id="cell-0">


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUN_DIR = '/content/drive/MyDrive/agentic_rl/runs/' + __import__('datetime').datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs(RUN_DIR, exist_ok=True)
print('RUN_DIR =', RUN_DIR)


## Setup: clone + env (idempotent)

In [ ]:
import os, subprocess

if not os.path.exists('/content/verl'):
    subprocess.run(['git', 'clone', '--depth', '1',
                     'https://github.com/verl-project/verl.git', '/content/verl'], check=True)
else:
    print('/content/verl already exists, skip clone')

print(subprocess.run(['git', '-C', '/content/verl', 'log', '--oneline', '-1'],
                      capture_output=True, text=True).stdout)


In [ ]:
import subprocess

# idempotent: skip uv sync if venv already has verl+vllm importable
check = subprocess.run(
    ['bash', '-c',
     'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp '
     'python3 -c "import verl, vllm, torch; print(\'OK\', torch.cuda.is_available())" 2>&1'],
    capture_output=True, text=True
)
print(check.stdout[-1000:])
need_sync = 'OK True' not in check.stdout

if need_sync:
    print('running uv sync (several GB, may take a while)...')
    r = subprocess.run(['bash', '-c',
        'cd /content/verl && uv sync --frozen --all-packages --extra vllm --extra fsdp'],
        capture_output=True, text=True)
    print(r.stdout[-3000:])
    print('STDERR:', r.stderr[-2000:])
    print('exit', r.returncode)
else:
    print('venv already OK, skip uv sync')


In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout)
r2 = subprocess.run(['bash', '-c',
    'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp python3 -c '
    '"import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))"'],
    capture_output=True, text=True)
print(r2.stdout)
print(r2.stderr[-500:])
assert 'True' in r2.stdout, 'CUDA not available, stop here'


## GSM8K preprocessing

In [ ]:
import subprocess, os
if not os.path.exists('/content/data/gsm8k/train.parquet'):
    r = subprocess.run(['bash', '-c',
        'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp python3 '
        'examples/data_preprocess/gsm8k.py --local_save_dir /content/data/gsm8k'],
        capture_output=True, text=True)
    print(r.stdout[-2000:])
    print(r.stderr[-1000:])
else:
    print('gsm8k parquet already exists, skip')

r = subprocess.run(['ls', '-la', '/content/data/gsm8k/'], capture_output=True, text=True)
print(r.stdout)


## Training: GRPO, Qwen2.5-0.5B-Instruct, single GPU, 100 steps, tool_agent(no tools)

10-step 파이프라인 검증(이전 노트북)은 `single_turn_agent`(전역 기본값)로 돌아
control arm 요건과 다른 경로였음 — 이번엔 `actor_rollout_ref.rollout.agent.default_agent_loop=tool_agent`
+`data.return_raw_chat=True`+`rollout.mode=async`로 tool arm과 동일한 agent_loop 경로 사용.
tool_config_path를 주지 않으면 tools=[]로 tool_agent가 동작(hermes parser는 여전히
활성 — `<tool_call>` 환각 생성 시 "Unknown function" 응답을 받고 재생성하므로
`num_turns/mean`이 2.0보다 커질 수 있음, 그 경우도 그대로 기록).

**판정기준(falsifiable, 사전 확정)**: `test_freq=10`으로 step10~100 strict val
측정, 마지막 3개 포인트(step80/90/100)가 각각 step0 대비 +0.05 이상이고 비감소면
"학습 진행"으로 판정. 결과 해석은 셀11에서 자동 계산.

`MAX_RESPONSE_LENGTH=1024` 유지. `SAVE_FREQ=-1`(체크포인트 미저장, 로그만 저장).
`trainer.val_before_train=True`로 step0 val을 자동 확보(판정기준의 기준점).
</cell id="cell-8">


In [ ]:
import subprocess, os, time, re

env = os.environ.copy()
env.update({
    'MODEL_PATH': 'Qwen/Qwen2.5-0.5B-Instruct',
    'TRAIN_FILE': '/content/data/gsm8k/train.parquet',
    'TEST_FILE': '/content/data/gsm8k/test.parquet',
    'NGPUS_PER_NODE': '1',
    'ROLLOUT_TP': '1',
    'ROLLOUT_GPU_MEM_UTIL': '0.5',
    'ROLLOUT_N': '5',
    'TRAIN_BATCH_SIZE': '20',
    'PPO_MINI_BATCH_SIZE': '10',
    'PPO_MICRO_BATCH_SIZE_PER_GPU': '2',
    'LOG_PROB_MICRO_BATCH_SIZE_PER_GPU': '2',
    'MAX_PROMPT_LENGTH': '512',
    'MAX_RESPONSE_LENGTH': '1024',
    'ACTOR_LR': '1e-6',
    'PROJECT_NAME': 'agentic_rl_exec',
    'EXPERIMENT_NAME': 'gsm8k_grpo_qwen25_0_5b_control_100step',
    'SAVE_FREQ': '-1',
    'TEST_FREQ': '10',
    'TOTAL_EPOCHS': '1',
})

val_dump_dir = os.path.join(RUN_DIR, 'val_dump')
log_path = os.path.join(RUN_DIR, 'train.log')
# set -o pipefail: tee가 항상 exit 0을 주므로 pipefail 없으면 학습이 죽어도
# proc.returncode가 0으로 나와 셀11이 미완주 로그를 완주로 오판할 위험이 있음.
cmd = (
    'set -o pipefail; cd /content/verl && bash examples/grpo_trainer/run_qwen3_4b_fsdp.sh '
    'trainer.logger=\'["console"]\' '
    'trainer.total_training_steps=100 '
    'trainer.val_before_train=True '
    f'trainer.validation_data_dir={val_dump_dir} '
    'actor_rollout_ref.rollout.mode=async '
    'actor_rollout_ref.rollout.agent.default_agent_loop=tool_agent '
    'data.return_raw_chat=True '
    f'2>&1 | tee {log_path}'
)
print('$', cmd)

# Popen(백그라운드) + 로그 폴링: step5 도달 시 timing_s/step 출력하고 계속 진행.
# (셀 전체는 학습 끝날 때까지 여기서 블로킹 — Run all 전제와 동일하게 동작)
proc = subprocess.Popen(['bash', '-c', cmd], env=env)

# 앵커(^)를 안 씀 — TaskRunner가 @ray.remote 액터라 Ray가 stdout에
# "(TaskRunnerV1 pid=NNNN) " 접두어를 붙일 가능성이 있음(미검증, 코드로는
# 확인 불가한 런타임 동작). 접두어가 붙어도 매칭되도록 방어적으로 작성.
step_pat = re.compile(r'step:(\d+)')
timing_pat = re.compile(r'timing_s/step:([0-9.]+)')

step5_reported = False
while proc.poll() is None:
    time.sleep(15)
    if not os.path.exists(log_path):
        continue
    with open(log_path) as f:
        log_tail = f.read()
    steps_seen = [int(m) for m in step_pat.findall(log_tail)]
    if not step5_reported and steps_seen and max(steps_seen) >= 5:
        timings = timing_pat.findall(log_tail)
        if timings:
            # step1은 warmup(컴파일 등)으로 느릴 수 있어 제외, step2~6 평균 — 하한 추정치.
            sample = timings[1:6] if len(timings) > 5 else timings
            avg = sum(float(t) for t in sample) / len(sample)
            print(f'[step5 체크포인트] step2~6 평균 timing_s/step={avg:.1f}s (하한 추정), '
                  f'100step 학습만 ≈ {avg*100/60:.1f}분 + val 11회 별도 소요 (계속 진행 중)')
        else:
            print('[step5 체크포인트] timing_s/step 로그 패턴 미발견(포맷 미검증) — 계속 진행')
        step5_reported = True
    # step5 확인 후엔 폴링 중단하고 대기만 — Drive FUSE 파일을 계속 재읽기하지 않음.
    if step5_reported:
        proc.wait()
        break

if proc.poll() is None:
    proc.wait()
print('exit', proc.returncode, '(pipefail 적용 — 0이 아니면 학습 실패, 셀11 결과 신뢰 불가)')
</cell id="cell-9">


## Results summary + env snapshot to Drive

In [ ]:
import subprocess, re, os

log_path = os.path.join(RUN_DIR, 'train.log')
with open(log_path) as f:
    log = f.read()

# 정확한 key 형식(verl/trainer/ppo/ray_trainer.py:734-738, 소스 확인 완료):
#   val-core/{data_source}/{acc|reward}/{mean|maj|best}@{n_max}
#   data_source="openai/gsm8k"(examples/data_preprocess/gsm8k.py:47), n_max=ROLLOUT_N=5
# 콘솔 print는 순수 "step:N - key:value - ..." 형태(aggregate_logger.py concat_dict_to_str)지만
# TaskRunner가 @ray.remote 액터라 Ray가 "(TaskRunnerV1 pid=NNNN) " 접두어를 stdout에
# 붙일 가능성 있음(미검증) — 정규식에 앵커(^) 안 씀.
for pat in [r'val-core/openai/gsm8k/[^\s]+', r'val-aux/num_turns/mean:[0-9.]+',
            r'critic/score/mean:[0-9.]+', r'step:\d+.*']:
    matches = re.findall(pat, log)
    print(f'--- {pat} ({len(matches)} matches) ---')
    for m in matches[:15]:
        print(m[:300])
    print()

# 완주 확인: exit code(pipefail 적용됨) + val 포인트 개수(step0 val_before_train 포함 11개:
# step0, 10, 20, ..., 100) + 마지막 step이 100인지.
exit_ok = proc.returncode == 0
final_steps = [int(m) for m in re.findall(r'step:(\d+)', log)]
reached_100 = bool(final_steps) and max(final_steps) >= 100

strict_vals = [float(v) for v in re.findall(r'val-core/openai/gsm8k/(?:acc|reward)/mean@\d+:([0-9.]+)', log)]

print('=== 완주 확인 ===')
print(f'exit_code={proc.returncode}(0이어야 정상), 마지막 step={max(final_steps) if final_steps else None}, '
      f'val 포인트 수={len(strict_vals)}(11개여야 함: step0+10,20..100)')

print()
print('=== 판정기준 자동계산(사전 확정: step0 대비 +0.05 이상, 마지막 3포인트 비감소) ===')
if not exit_ok:
    print('학습이 비정상 종료됨(exit != 0) — 판정 보류. train.log 끝부분 직접 확인 필요.')
elif not reached_100 or len(strict_vals) < 11:
    print(f'100step 미완주 또는 val 포인트 부족(reached_100={reached_100}, '
          f'val개수={len(strict_vals)}) — 판정 보류.')
elif len(strict_vals) < 4:
    print(f'val 포인트 {len(strict_vals)}개 — 위 raw dump를 직접 확인, 정규식이 실제 key와 다를 수 있음.')
else:
    step0 = strict_vals[0]
    last3 = strict_vals[-3:]
    threshold = step0 + 0.05
    meets_threshold = all(v >= threshold for v in last3)
    non_decreasing = all(last3[i] <= last3[i+1] for i in range(len(last3)-1))
    verdict = "학습 진행" if (meets_threshold and non_decreasing) else "미달 — 옵션 1개만 변경(+100step 또는 lr 1e-6→3e-6)"
    print(f'step0={step0:.4f}, 임계값(step0+0.05)={threshold:.4f}')
    print(f'마지막 3개 val={last3}')
    print(f'임계값 이상: {meets_threshold}, 비감소: {non_decreasing}')
    print(f'판정: {verdict}')

# env snapshot
freeze = subprocess.run(['bash', '-c',
    'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp pip freeze'],
    capture_output=True, text=True).stdout
with open(os.path.join(RUN_DIR, 'pip_freeze.txt'), 'w') as f:
    f.write(freeze)

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
with open(os.path.join(RUN_DIR, 'nvidia-smi.txt'), 'w') as f:
    f.write(smi)

print('Saved to', RUN_DIR)
print(subprocess.run(['ls', '-la', RUN_DIR], capture_output=True, text=True).stdout)
</cell id="cell-11">


## 끝났으면

- `RUN_DIR` 아래 (`train.log`, `pip_freeze.txt`, `nvidia-smi.txt`)를 Drive에서 확인.
- 셀11의 "판정기준 자동계산" 출력이 metric key 불일치로 실패하면, raw dump
  섹션(`val/test_score/...`, `critic/score/mean`)을 직접 눈으로 대조 — 정규식이
  실제 로그 포맷과 다를 수 있음(사전 실측 없이 작성됨, 확인 후 STATE에 정정 반영).
- `val-aux/num_turns/mean`이 2.0이 아니면 hermes parser 환각 여부 확인 필요.
- 이 세션은 종료해도 됨 — 결과는 Drive에 남아있음.
- 다음 raw 결과는 로컬 WSL에서 `/mnt/c/.../Google Drive/agentic_rl/runs/...` 경로로 확인 가능(Google Drive Desktop 동기화 시).
- 완주 후 다음 착수점: 이 control arm 결과를 STATE/JOURNAL에 반영 → tool arm 노트북(SandboxTool timeout 패치 포함) 작성.
</cell id="cell-12">
